In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import sequence

In [19]:
##Load the word index mapping from the IMDB dataset. This mapping is used to convert words in the reviews to their corresponding integer indices.
word_index = imdb.get_word_index()
reverse_word_index = {value: key for (key, value) in word_index.items()} ##The reverse mapping is created to convert integer indices back to words, which can be useful for interpreting the reviews and the model's predictions.

In [20]:
##Load the pre-trained model
model = load_model('simple_rnn_imdb.h5') ##The pre-trained model is loaded from the file 'simplernn_model.h5' using the load_model function. This allows us to use the model for making predictions on new reviews without having to retrain it.
model.summary() ##The summary method is called to display the architecture of the loaded model, including the number of parameters in each layer and the total number of parameters in the model. This is useful for understanding the structure of the model and for debugging purposes.

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
##Helper function to decode the integer-encoded reviews back to text
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review]) ##The function takes an integer-encoded review as input and returns the corresponding text review by mapping each integer back to its corresponding word using the reverse_word_index. The get method is used to handle cases where an index might not be found in the reverse mapping, returning a '?' in such cases.

#Function to preprocess the input review and make a prediction
def preprocess_text(text):
    words = text.lower().split() ##The input review is converted to lowercase and split into individual words.
    encoded_review = [word_index.get(word, 2)+3 for word in words] ##Each word in the review is converted to its corresponding integer index using the word_index mapping. If a word is not found in the mapping, it is assigned a default index of 2 (which typically represents an unknown token).
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500) ##The encoded review is padded to a maximum length of 500 using the pad_sequences function. This ensures that the input to the model has a consistent shape, which is necessary for making predictions.
    return padded_review

In [26]:
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    score = prediction[0][0] ##The predicted score is extracted from the model's output. This score represents the confidence of the model in predicting a positive sentiment, with values closer to 1 indicating a stronger positive sentiment and values closer to 0 indicating a stronger negative sentiment.
    if score >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, prediction

In [27]:
##Example usage
example_review = "This movie was fantastic! I really enjoyed it. The acting was great and the plot was very engaging."
sentiment, score = predict_sentiment(example_review)
print(f"Review: {example_review}")
print(f"Predicted Sentiment: {sentiment} (Score: {score[0][0]:.4f})") ##The example review is passed to the predict_sentiment function, and the predicted sentiment along with the confidence score is printed. The score indicates how confident the model is in its prediction, with values closer to 1 indicating a stronger positive sentiment and values closer to 0 indicating a stronger negative sentiment.

1/1 [==============================] - 0s 30ms/step
Review: This movie was fantastic! I really enjoyed it. The acting was great and the plot was very engaging.
Predicted Sentiment: Positive (Score: 0.9356)
